In [1]:
import pandas as pd
arrests = pd.read_parquet('data/arrests-latest.parquet')

In [30]:
arrests
columns_to_keep = ["apprehension_date", "apprehension_type", "apprehension_aor", 'duplicate_likely', 'unique_identifier', "apprehension_criminality", 'birth_year','gender','citizenship_country']
arrests = arrests[columns_to_keep]
arrests = arrests.rename(columns={'apprehension_aor': 'aor_nam', 'apprehension_date': 'Apprehension Date', 'apprehension_criminality': 'Apprehension Criminality', 'birth_year': 'Birth Year', 'gender': 'Gender', 'citizenship_country': 'Citizenship Country'})
arrests = arrests.replace(to_replace="1 Convicted Criminal", value="Convicted")
arrests = arrests.replace(to_replace="2 Pending Criminal Charges", value="Pending Charges")
arrests = arrests.replace(to_replace="3 Other Immigration Violator", value="No Criminal Charges")
arrests['aor_nam'] = arrests['aor_nam'].str.replace(' Area of Responsibility', '', regex=False)
arrests['Apprehension MonthYear'] = pd.to_datetime(arrests['Apprehension Date']).dt.strftime('%Y-%m')
arrests['Apprehension Date'] = pd.to_datetime(arrests['Apprehension Date'])
print(arrests['duplicate_likely'].value_counts()) # 16085
arrests = arrests[arrests['duplicate_likely'] == False] # 16085
arrests['Age'] = arrests['Apprehension Date'].dt.year - arrests['Birth Year'].dropna().astype(int)
arrests['Age Group'] = pd.cut(arrests['Age'], 
                                     bins=[0, 18, 25, 35, 45, 55, 65, 100], 
                                     labels=['0-17', '18-24', '25-34', '35-44', '45-54', '55-64', '65+'],
                                     right=False)
# apprehension type only available starting in August 2025

duplicate_likely
False    683036
True      16085
Name: count, dtype: int64


In [45]:
arrests_collateral = arrests[arrests['Apprehension Date'] >= '2025-08-08']
arrests_collateral.head()

,Apprehension Date,apprehension_type,aor_nam,duplicate_likely,unique_identifier,Apprehension Criminality,Birth Year,Gender,Citizenship Country,Apprehension MonthYear,Age,Age Group
466910,2025-08-08,Targeted,Atlanta,False,fe7b8d2fee0b890741acc569506ad11279a6b34b,Pending Charges,1978.0,Male,NICARAGUA,2025-08,47.0,45-54
466911,2025-08-08,Targeted,Phoenix,False,b0de6735ac05fd47d81e1e87b8d70970f6ce1e00,Pending Charges,2004.0,Male,DEM REP OF THE CONGO,2025-08,21.0,18-24
466912,2025-08-08,Targeted,Houston,False,ce4e0ae7e22937f6c602ee3481563b8042a6ee48,Convicted,1995.0,Male,HONDURAS,2025-08,30.0,25-34
466913,2025-08-08,Targeted,Houston,False,c6baae0bebda2dbeb360da4070ce8f8c5723e7d0,No Criminal Charges,1980.0,Male,GUATEMALA,2025-08,45.0,45-54
466914,2025-08-08,Targeted,Houston,False,cc54350cbd0a3cd5c1500dad687b2a9ef9a41c53,Convicted,1967.0,Female,MEXICO,2025-08,58.0,55-64


In [64]:
# Recode 'None' in apprehension_type to 'Unknown'
arrests_collateral['apprehension_type'] = arrests_collateral['apprehension_type'].fillna('Unknown')

# Group by date, aor_nam, and apprehension_type, and count arrests
daily_counts = arrests_collateral.groupby(['Apprehension Date', 'aor_nam', 'apprehension_type']).size().reset_index(name='count')

# Calculate 7-day rolling average for each (aor_nam, apprehension_type) group
rolling_averages_df = daily_counts.copy()
rolling_averages_df = rolling_averages_df.sort_values(['aor_nam', 'apprehension_type', 'Apprehension Date'])
rolling_averages_df['rolling_avg_7d'] = rolling_averages_df.groupby(['aor_nam', 'apprehension_type'])['count'].rolling(window=7, min_periods=1).mean().reset_index(level=[0,1], drop=True).round(1)

# Calculate national totals grouped by apprehension_type
national_daily_totals = daily_counts.groupby(['Apprehension Date', 'apprehension_type'])['count'].sum().reset_index(name='count')
national_daily_totals = national_daily_totals.sort_values(['apprehension_type', 'Apprehension Date'])
national_daily_totals['rolling_avg_7d'] = national_daily_totals.groupby('apprehension_type')['count'].rolling(window=7, min_periods=1).mean().reset_index(level=0, drop=True).round(1)
national_daily_totals['aor_nam'] = 'National'

# Append national rows to the bottom
rolling_averages_df = pd.concat([rolling_averages_df[['Apprehension Date', 'aor_nam', 'apprehension_type', 'count', 'rolling_avg_7d']], 
                                  national_daily_totals[['Apprehension Date', 'aor_nam', 'apprehension_type', 'count', 'rolling_avg_7d']]], 
                                 ignore_index=True)

/var/folders/1l/yh12s4qx29z26j7mg26bfby80000gn/T/ipykernel_90113/510513009.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  arrests_collateral['apprehension_type'] = arrests_collateral['apprehension_type'].fillna('Unknown')


In [66]:
rolling_averages_df.to_csv('data/apprehension_type_rolling_averages.csv', index=False)

In [ ]:
print("=== NATIONAL ROLLING AVERAGE (ARRESTS_COLLATERAL) ===\n")

# Aggregate collateral arrests by date nationally
national_daily = arrests_collateral.groupby('Apprehension Date').size().reset_index(name='count')
national_daily['Apprehension Date'] = pd.to_datetime(national_daily['Apprehension Date'])
national_daily = national_daily.sort_values('Apprehension Date')

# Calculate 7-day rolling average
national_daily['rolling_avg_7d'] = national_daily['count'].rolling(window=7, min_periods=1).mean().round(1)

=== NATIONAL ROLLING AVERAGE (ARRESTS_COLLATERAL) ===

Total collateral arrests (2025-08-08 onward): 231,915
Date range: 2025-08-08 to 2026-09-17

Daily arrests - last 20 days:
Apprehension Date  count  rolling_avg_7d
       2026-02-21    731          1000.1
       2026-02-22    526           995.1
       2026-02-23    938          1010.6
       2026-02-24   1245          1021.7
       2026-02-25   1357          1036.7
       2026-02-26   1252          1032.0
       2026-02-27   1104          1021.9
       2026-02-28    721          1020.4
       2026-03-01    596          1030.4
       2026-03-02   1066          1048.7
       2026-03-03   1270          1052.3
       2026-03-04   1313          1046.0
       2026-03-05   1183          1036.1
       2026-03-06   1093          1034.6
       2026-03-07    647          1024.0
       2026-03-08    500          1010.3
       2026-03-09   1015          1003.0
       2026-03-10   1209           994.3
       2026-03-11      1           806.9
   

In [38]:
metrosurge = arrests[(arrests['Apprehension Date'] >= '2025-12-01') & (arrests['Apprehension Date'] < '2026-03-01') & (arrests['aor_nam'] == 'St. Paul')]
metrosurge.head()

,Apprehension Date,apprehension_type,aor_nam,duplicate_likely,unique_identifier,Apprehension Criminality,Birth Year,Gender,Citizenship Country,Apprehension MonthYear,Age,Age Group
594745,2025-12-01,Targeted,St. Paul,False,37d02681d544badf82275665e4054102b8700345,Pending Charges,1997.0,Male,COLOMBIA,2025-12,28.0,25-34
594764,2025-12-01,Targeted,St. Paul,False,f7c5e112e55bb84b96f5a49c892bc804187d7244,Pending Charges,1995.0,Male,GUATEMALA,2025-12,30.0,25-34
594776,2025-12-01,Targeted,St. Paul,False,9eac812d38a835db1e7daa758bc4d91fee574b44,Convicted,1997.0,Male,MEXICO,2025-12,28.0,25-34
594833,2025-12-01,Targeted,St. Paul,False,35d7a13cbe46e7f578cd220c5bc1fa43cf9c7bbc,Pending Charges,1988.0,Male,EL SALVADOR,2025-12,37.0,35-44
594834,2025-12-01,Targeted,St. Paul,False,b38e0d2d32332fb2e64790e8abf3034effe0df67,No Criminal Charges,1997.0,Male,SOMALIA,2025-12,28.0,25-34


In [39]:
# Descriptive statistics for apprehension_type in metrosurge
print("=== DESCRIPTIVE STATS FOR APPREHENSION_TYPE ===\n")

# Value counts
value_counts = metrosurge['apprehension_type'].value_counts(dropna=False)
print("Value counts (including NaN):")
print(value_counts)
print()

# Percentages
percentages = metrosurge['apprehension_type'].value_counts(normalize=True, dropna=False) * 100
print("Percentages (including NaN):")
print(percentages.round(2).astype(str) + '%')
print()

# Summary stats
total = len(metrosurge)
non_missing = metrosurge['apprehension_type'].notna().sum()
missing = metrosurge['apprehension_type'].isna().sum()
print(f"Total records: {total}")
print(f"Non-missing: {non_missing} ({(non_missing/total*100):.2f}%)")
print(f"Missing: {missing} ({(missing/total*100):.2f}%)")

=== DESCRIPTIVE STATS FOR APPREHENSION_TYPE ===

Value counts (including NaN):
apprehension_type
Targeted      3295
Collateral    1193
None             1
Name: count, dtype: int64

Percentages (including NaN):
apprehension_type
Targeted       73.4%
Collateral    26.58%
None           0.02%
Name: proportion, dtype: object

Total records: 4489
Non-missing: 4488 (99.98%)
Missing: 1 (0.02%)


In [40]:
# Descriptive statistics for apprehension_type overall
print("=== DESCRIPTIVE STATS FOR APPREHENSION_TYPE ===\n")

# Value counts
value_counts = arrests_collateral['apprehension_type'].value_counts(dropna=False)
print("Value counts (including NaN):")
print(value_counts)
print()

# Percentages
percentages = arrests_collateral['apprehension_type'].value_counts(normalize=True, dropna=False) * 100
print("Percentages (including NaN):")
print(percentages.round(2).astype(str) + '%')
print()

# Summary stats
total = len(arrests_collateral)
non_missing = arrests_collateral['apprehension_type'].notna().sum()
missing = arrests_collateral['apprehension_type'].isna().sum()
print(f"Total records: {total}")
print(f"Non-missing: {non_missing} ({(non_missing/total*100):.2f}%)")
print(f"Missing: {missing} ({(missing/total*100):.2f}%)")

=== DESCRIPTIVE STATS FOR APPREHENSION_TYPE ===

Value counts (including NaN):
apprehension_type
Targeted      178937
Collateral     51676
None            1302
Name: count, dtype: int64

Percentages (including NaN):
apprehension_type
Targeted      77.16%
Collateral    22.28%
None           0.56%
Name: proportion, dtype: object

Total records: 231915
Non-missing: 230613 (99.44%)
Missing: 1302 (0.56%)


In [41]:
print("=== APPREHENSION_TYPE MISSINGNESS IN ARRESTS_COLLATERAL ===\n")

total = len(arrests_collateral)
missing = arrests_collateral['apprehension_type'].isna().sum()
missing_pct = (missing / total) * 100
present = arrests_collateral['apprehension_type'].notna().sum()

print(f"Total records: {total:,}")
print(f"Missing apprehension_type: {missing:,} ({missing_pct:.2f}%)")
print(f"Present apprehension_type: {present:,} ({100-missing_pct:.2f}%)")

print(f"\n\nValue counts (non-missing):")
print(arrests_collateral['apprehension_type'].value_counts(dropna=False))


=== APPREHENSION_TYPE MISSINGNESS IN ARRESTS_COLLATERAL ===

Total records: 231,915
Missing apprehension_type: 1,302 (0.56%)
Present apprehension_type: 230,613 (99.44%)


Value counts (non-missing):
apprehension_type
Targeted      178937
Collateral     51676
None            1302
Name: count, dtype: int64


In [67]:
print("="*70)
print("BREAKDOWN BY APPREHENSION TYPE: CRIMINALITY & AGE GROUP")
print("="*70)

# Filter to only Targeted and Collateral (exclude Unknown/missing)
arrests_clean = arrests_collateral[arrests_collateral['apprehension_type'].isin(['Targeted', 'Collateral'])].copy()

print(f"\nTotal records (excluding Unknown): {len(arrests_clean):,}")
print(f"Targeted: {len(arrests_clean[arrests_clean['apprehension_type'] == 'Targeted']):,}")
print(f"Collateral: {len(arrests_clean[arrests_clean['apprehension_type'] == 'Collateral']):,}")

# APPREHENSION CRIMINALITY BREAKDOWN
print("\n" + "="*70)
print("APPREHENSION CRIMINALITY")
print("="*70)

criminality_breakdown = arrests_clean.groupby(['apprehension_type', 'Apprehension Criminality']).size().reset_index(name='count')
criminality_pct = criminality_breakdown.pivot_table(index='Apprehension Criminality', 
                                                     columns='apprehension_type', 
                                                     values='count', 
                                                     fill_value=0)
criminality_pct_pct = criminality_pct.div(criminality_pct.sum(axis=0), axis=1) * 100

print("\nCounts:")
print(criminality_pct.astype(int).to_string())

print("\nPercentage Distribution (%):")
print(criminality_pct_pct.round(1).to_string())

# AGE GROUP BREAKDOWN
print("\n" + "="*70)
print("AGE GROUP")
print("="*70)

age_breakdown = arrests_clean.groupby(['apprehension_type', 'Age Group']).size().reset_index(name='count')
age_pct = age_breakdown.pivot_table(index='Age Group', 
                                     columns='apprehension_type', 
                                     values='count', 
                                     fill_value=0)
age_pct_pct = age_pct.div(age_pct.sum(axis=0), axis=1) * 100

print("\nCounts:")
print(age_pct.astype(int).to_string())

print("\nPercentage Distribution (%):")
print(age_pct_pct.round(1).to_string())

# SUMMARY STATISTICS
print("\n" + "="*70)
print("KEY DIFFERENCES")
print("="*70)

print("\nCriminality:")
for crim in criminality_pct_pct.index:
    targeted = criminality_pct_pct.loc[crim, 'Targeted']
    collateral = criminality_pct_pct.loc[crim, 'Collateral']
    diff = targeted - collateral
    print(f"  {crim:30s}: Targeted {targeted:5.1f}% | Collateral {collateral:5.1f}% | Diff: {diff:+6.1f}pp")

print("\nAge Group:")
for age in age_pct_pct.index:
    targeted = age_pct_pct.loc[age, 'Targeted']
    collateral = age_pct_pct.loc[age, 'Collateral']
    diff = targeted - collateral
    print(f"  {str(age):30s}: Targeted {targeted:5.1f}% | Collateral {collateral:5.1f}% | Diff: {diff:+6.1f}pp")


BREAKDOWN BY APPREHENSION TYPE: CRIMINALITY & AGE GROUP

Total records (excluding Unknown): 230,613
Targeted: 178,937
Collateral: 51,676

APPREHENSION CRIMINALITY

Counts:
apprehension_type         Collateral  Targeted
Apprehension Criminality                      
Convicted                      10368     58680
No Criminal Charges            31455     63691
Pending Charges                 9853     56566

Percentage Distribution (%):
apprehension_type         Collateral  Targeted
Apprehension Criminality                      
Convicted                       20.1      32.8
No Criminal Charges             60.9      35.6
Pending Charges                 19.1      31.6

AGE GROUP

Counts:
apprehension_type  Collateral  Targeted
Age Group                              
0-17                      697      2603
18-24                    8699     26392
25-34                   17573     62904
35-44                   14058     50193
45-54                    7843     26891
55-64                    239

/var/folders/1l/yh12s4qx29z26j7mg26bfby80000gn/T/ipykernel_90113/463992086.py:35: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  age_breakdown = arrests_clean.groupby(['apprehension_type', 'Age Group']).size().reset_index(name='count')
/var/folders/1l/yh12s4qx29z26j7mg26bfby80000gn/T/ipykernel_90113/463992086.py:36: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  age_pct = age_breakdown.pivot_table(index='Age Group',


In [68]:
print("\n" + "="*70)
print("SUMMARY: TARGETED vs COLLATERAL")
print("="*70)

print(f"""
KEY FINDINGS:

CRIMINALITY PROFILE:
  • Targeted arrests: {criminality_pct_pct.loc['Convicted', 'Targeted']:.1f}% Convicted | {criminality_pct_pct.loc['Pending Charges', 'Targeted']:.1f}% Pending | {criminality_pct_pct.loc['No Criminal Charges', 'Targeted']:.1f}% No Charges
  • Collateral arrests: {criminality_pct_pct.loc['Convicted', 'Collateral']:.1f}% Convicted | {criminality_pct_pct.loc['Pending Charges', 'Collateral']:.1f}% Pending | {criminality_pct_pct.loc['No Criminal Charges', 'Collateral']:.1f}% No Charges
  
  → Targeted arrests have MORE criminal history/charges
  → Collateral arrests are more likely to have NO criminal charges

AGE PROFILE:
  • Targeted median age group: {age_pct_pct['Targeted'].idxmax()} (peak {age_pct_pct.loc[age_pct_pct['Targeted'].idxmax(), 'Targeted']:.1f}%)
  • Collateral median age group: {age_pct_pct['Collateral'].idxmax()} (peak {age_pct_pct.loc[age_pct_pct['Collateral'].idxmax(), 'Collateral']:.1f}%)
  
  → Distributions indicate different demographic targeting patterns
""")



SUMMARY: TARGETED vs COLLATERAL

KEY FINDINGS:

CRIMINALITY PROFILE:
  • Targeted arrests: 32.8% Convicted | 31.6% Pending | 35.6% No Charges
  • Collateral arrests: 20.1% Convicted | 19.1% Pending | 60.9% No Charges

  → Targeted arrests have MORE criminal history/charges
  → Collateral arrests are more likely to have NO criminal charges

AGE PROFILE:
  • Targeted median age group: 25-34 (peak 35.2%)
  • Collateral median age group: 25-34 (peak 34.0%)

  → Distributions indicate different demographic targeting patterns



In [69]:
print("\nDETAILED BREAKDOWN:\n")

print("Apprehension Criminality (% within each type):")
print(criminality_pct_pct.round(1))

print("\n\nAge Group (% within each type):")
print(age_pct_pct.round(1))

print("\n\nTop differentiators (largest gaps between Targeted vs Collateral):")
print("\nCriminality gaps (percentage points):")
crim_gaps = (criminality_pct_pct['Targeted'] - criminality_pct_pct['Collateral']).sort_values(ascending=False)
for crim, gap in crim_gaps.items():
    print(f"  {crim:30s}: {gap:+6.1f}pp")

print("\nAge group gaps (percentage points):")
age_gaps = (age_pct_pct['Targeted'] - age_pct_pct['Collateral']).sort_values(ascending=False)
for age, gap in age_gaps.items():
    print(f"  {str(age):30s}: {gap:+6.1f}pp")



DETAILED BREAKDOWN:

Apprehension Criminality (% within each type):
apprehension_type         Collateral  Targeted
Apprehension Criminality                      
Convicted                       20.1      32.8
No Criminal Charges             60.9      35.6
Pending Charges                 19.1      31.6


Age Group (% within each type):
apprehension_type  Collateral  Targeted
Age Group                              
0-17                      1.3       1.5
18-24                    16.8      14.7
25-34                    34.0      35.2
35-44                    27.2      28.1
45-54                    15.2      15.0
55-64                     4.6       4.6
65+                       0.8       0.9


Top differentiators (largest gaps between Targeted vs Collateral):

Criminality gaps (percentage points):
  Convicted                     :  +12.7pp
  Pending Charges               :  +12.5pp
  No Criminal Charges           :  -25.3pp

Age group gaps (percentage points):
  25-34                     